# PS5 pill PatchCore-lite training/export

This notebook runs the repository scripts. It does not download the dataset or model weights on your behalf. Keep the MVTec archive out of Git and review all dataset/model terms before use. Bank sizes below are experimental candidates only; select the shipped size from phone accuracy/latency measurements.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import subprocess
import sys

# Set these to the repository and extracted MVTec directory in your Colab Drive.
REPO = Path('/content/drive/MyDrive/PS5')
DATA_ROOT = REPO / 'ml' / 'data'
CATEGORY = 'pill'
SEED = 42
BANK_CANDIDATES = '256,1024,4096'

assert (REPO / 'ml' / 'scripts' / '00_prepare_splits.py').is_file()
assert (DATA_ROOT / CATEGORY / 'train' / 'good').is_dir()

In [ ]:
subprocess.run([
    'python', str(REPO / 'ml' / 'scripts' / '00_prepare_splits.py'),
    '--data-root', str(DATA_ROOT),
    '--output', str(REPO / 'ml' / 'splits' / f'{CATEGORY}.json'),
    '--category', CATEGORY,
    '--seed', str(SEED),
    '--stats-output', str(REPO / 'results' / 'dataset_stats.json'),
], check=True)

In [ ]:
subprocess.run([
    'python', str(REPO / 'ml' / 'scripts' / '01_train_patchcore_lite.py'),
    '--data-root', str(DATA_ROOT),
    '--split', str(REPO / 'ml' / 'splits' / f'{CATEGORY}.json'),
    '--output-dir', str(REPO / 'ml' / 'checkpoints' / CATEGORY),
    '--bank-sizes', BANK_CANDIDATES,
    '--seed', str(SEED),
], check=True)

Export and test one candidate with ONNX Runtime. This is a parity candidate only; the final memory-bank size still requires a phone accuracy/latency sweep. The ExecuTorch `.pte` exporter remains available as `02_export_vision_executorch.py`, but Windows host runtime parity fails even on a minimal XNNPACK model. Llama advisory remains ExecuTorch.

In [ ]:
candidate = BANK_CANDIDATES.split(',')[0]
checkpoint = REPO / 'ml' / 'checkpoints' / CATEGORY / f'patchcore_{CATEGORY}_bank_{candidate}.pt'
onnx_path = REPO / 'ml' / 'checkpoints' / CATEGORY / f'patchcore_{CATEGORY}_{candidate}.onnx'
subprocess.run([
    sys.executable, str(REPO / 'ml' / 'scripts' / '02_export_vision.py'),
    '--checkpoint', str(checkpoint), '--output', str(onnx_path),
], check=True)
subprocess.run([
    sys.executable, str(REPO / 'ml' / 'scripts' / '03_parity_test.py'),
    '--checkpoint', str(checkpoint), '--onnx', str(onnx_path),
    '--split', str(REPO / 'ml' / 'splits' / f'{CATEGORY}.json'),
    '--data-root', str(DATA_ROOT),
    '--output', str(REPO / 'results' / f'parity_{CATEGORY}_{candidate}.csv'),
], check=True)